In [3]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [4]:
df_sal = pd.read_csv(root/"data"/"raw"/"ipeds"/"CSV_422026-190.csv")

In [5]:
df_sal_clean = df_sal.copy()

# Rename columns (short + usable)
df_sal_clean = df_sal_clean.rename(columns={
    'DRVF2024.Salaries and wages for instruction as a percent of total expenses for instruction (GASB)': 'instruction_salary_pct',
    'DRVF2024.Salaries and wages for research as a percent of total expenses for research (GASB)': 'research_salary_pct',
    'DRVF2024.Salaries and wages for academic support as a percent of total expenses for academic support (GASB)': 'academic_support_salary_pct',
    'DRVF2024.Salaries and wages for student services as a percent of total expenses for student services (GASB)': 'student_services_salary_pct',
})

# Keep only relevant columns
df_sal_clean = df_sal_clean[
    [
        'unitid',
        'year',
        'instruction_salary_pct',
        'academic_support_salary_pct',
        'student_services_salary_pct',
        # optional:
        'research_salary_pct'
    ]
]

# Ensure numeric
for col in df_sal_clean.columns:
    if col not in ['unitid', 'year']:
        df_sal_clean[col] = pd.to_numeric(df_sal_clean[col], errors='coerce')

In [6]:
print(df_sal_clean.head())
print(df_sal_clean.info())

   unitid  year  instruction_salary_pct  academic_support_salary_pct  \
0  132374  2024                    42.0                         54.0   
1  132408  2024                     NaN                          NaN   
2  132471  2024                     NaN                          NaN   
3  132602  2024                     NaN                          NaN   
4  132657  2024                     NaN                          NaN   

   student_services_salary_pct  research_salary_pct  
0                         69.0                  NaN  
1                          NaN                  NaN  
2                          NaN                  NaN  
3                          NaN                  NaN  
4                          NaN                  NaN  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   unitid            

In [7]:
df_istaff = pd.read_csv(root/"data"/"raw"/"ipeds"/"CSV_422026-852.csv")
df_gen = pd.read_csv(root/"data"/"raw"/"ipeds"/"CSV_422026-647.csv")
df_cip_completers = pd.read_csv(root/"data"/"raw"/"ipeds"/"CSV_422026-933.csv")

In [8]:
df_gen_clean = df_gen.copy()

df_gen_clean = df_gen_clean.rename(columns={
    'IC2020.Institution does not accept dual, credit for life, or AP credits': 'no_ap_credit',
    'IC2020.Study abroad': 'study_abroad',
    'IC2020.Academic/career counseling service': 'career_counseling',
    'IC2020.Employment services for students': 'employment_services',
    'IC2020.Placement services for completers': 'placement_services',

    'DRVF2020.Instruction expenses as a percent of total core expenses (GASB)': 'instruction_expense_pct',
    'DRVF2020.Research expenses as a percent of total core expenses (GASB)': 'research_expense_pct',
    'DRVF2020.Student service expenses as a percent of total core expenses (GASB)': 'student_service_expense_pct',
    'DRVF2020.Endowment assets (year end) per FTE enrollment (GASB)': 'endowment_per_fte',
    'DRVF2020.Equity ratio (GASB)': 'equity_ratio',

    'DRVHR2020.Total FTE staff': 'total_fte_staff',
    'DRVHR2020.Instructional, research and public service FTE': 'irps_fte',
    'DRVHR2020.Instructional FTE': 'instructional_fte',
    'DRVHR2020.Research FTE': 'research_fte',

    'DRVEF122020.Total 12-month unduplicated headcount': 'total_headcount',
})

keep_cols = [
    'unitid', 'year',
    'no_ap_credit', 'study_abroad', 'career_counseling',
    'employment_services', 'placement_services',
    'instruction_expense_pct', 'research_expense_pct',
    'student_service_expense_pct', 'endowment_per_fte',
    'equity_ratio', 'total_fte_staff', 'irps_fte',
    'instructional_fte', 'research_fte', 'total_headcount'
]

df_gen_clean = df_gen_clean[keep_cols].copy()

binary_cols = [
    'no_ap_credit', 'study_abroad', 'career_counseling',
    'employment_services', 'placement_services'
]

numeric_cols = [c for c in df_gen_clean.columns if c not in ['unitid', 'year'] + binary_cols]

def encode_service(col):
    return (
        col.astype(str)
        .str.strip()
        .str.lower()
        .map({
            'yes': 1,
            'implied no': 0
        })
    )

for c in binary_cols:
    df_gen_clean[c] = encode_service(df_gen_clean[c])

for c in numeric_cols:
    df_gen_clean[c] = pd.to_numeric(df_gen_clean[c], errors='coerce')

# protect against divide-by-zero
den = df_gen_clean['total_headcount'].replace({0: np.nan})
staff_den = df_gen_clean['total_fte_staff'].replace({0: np.nan})

df_gen_clean['staff_per_student'] = df_gen_clean['total_fte_staff'] / den
df_gen_clean['instructional_fte_per_student'] = df_gen_clean['instructional_fte'] / den
df_gen_clean['irps_fte_per_student'] = df_gen_clean['irps_fte'] / den

df_gen_clean['instructional_share_of_staff'] = df_gen_clean['instructional_fte'] / staff_den
df_gen_clean['research_share_of_staff'] = df_gen_clean['research_fte'] / staff_den

# optional: drop raw counts after engineering
df_gen_clean = df_gen_clean.drop(columns=[
    'total_fte_staff', 'irps_fte', 'instructional_fte', 'research_fte', 'total_headcount'
])

print(df_gen_clean.columns)
print(df_gen_clean.shape)

Index(['unitid', 'year', 'no_ap_credit', 'study_abroad', 'career_counseling',
       'employment_services', 'placement_services', 'instruction_expense_pct',
       'research_expense_pct', 'student_service_expense_pct',
       'endowment_per_fte', 'equity_ratio', 'staff_per_student',
       'instructional_fte_per_student', 'irps_fte_per_student',
       'instructional_share_of_staff', 'research_share_of_staff'],
      dtype='object')
(336, 17)


In [9]:
for c in binary_cols:
    print(c, df_gen_clean[c].mean())



no_ap_credit 0.4745222929936306
study_abroad 0.2070063694267516
career_counseling 0.9203821656050956
employment_services 0.6910828025477707
placement_services 0.8757961783439491


In [10]:
df_gen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 18 columns):
 #   Column                                                                        Non-Null Count  Dtype  
---  ------                                                                        --------------  -----  
 0   unitid                                                                        336 non-null    int64  
 1   institution name                                                              336 non-null    object 
 2   year                                                                          336 non-null    int64  
 3   IC2020.Institution does not accept dual, credit for life, or AP credits       314 non-null    object 
 4   IC2020.Study abroad                                                           314 non-null    object 
 5   IC2020.Academic/career counseling service                                     314 non-null    object 
 6   IC2020.Employment services for stu

In [11]:
df_gen_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 336 entries, 0 to 335
Data columns (total 17 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   unitid                         336 non-null    int64  
 1   year                           336 non-null    int64  
 2   no_ap_credit                   314 non-null    float64
 3   study_abroad                   314 non-null    float64
 4   career_counseling              314 non-null    float64
 5   employment_services            314 non-null    float64
 6   placement_services             314 non-null    float64
 7   instruction_expense_pct        89 non-null     float64
 8   research_expense_pct           89 non-null     float64
 9   student_service_expense_pct    89 non-null     float64
 10  endowment_per_fte              40 non-null     float64
 11  equity_ratio                   41 non-null     float64
 12  staff_per_student              312 non-null    flo

In [12]:
df_istaff.columns

Index(['unitid', 'institution name', 'year', 'SAL2020_IS.Academic rank',
       'SAL2020_IS.Instructional staff - total',
       'SAL2020_IS.Instructional staff on less than 9-month contract-total',
       'SAL2020_IS.Instructional staff on 9, 10, 11 or 12 month contract-total',
       'IDX_HR'],
      dtype='object')

In [13]:
df_istaff_clean = df_istaff.copy().rename(columns={
    'SAL2020_IS.Instructional staff - total': 'instructional_staff_total',
    'SAL2020_IS.Instructional staff on less than 9-month contract-total': 'instructional_staff_short_contract',
    'SAL2020_IS.Instructional staff on 9, 10, 11 or 12 month contract-total': 'instructional_staff_long_contract',
})

keep_cols = [
    'unitid',
    'year',
    'instructional_staff_total',
    'instructional_staff_short_contract',
    'instructional_staff_long_contract',
]

df_istaff_clean = df_istaff_clean[keep_cols].copy()

for c in [
    'instructional_staff_total',
    'instructional_staff_short_contract',
    'instructional_staff_long_contract'
]:
    df_istaff_clean[c] = pd.to_numeric(df_istaff_clean[c], errors='coerce')

# THIS is the real fix
df_istaff_clean = (
    df_istaff_clean
    .groupby(['unitid', 'year'], as_index=False)
    .agg({
        'instructional_staff_total': 'sum',
        'instructional_staff_short_contract': 'sum',
        'instructional_staff_long_contract': 'sum',
    })
)

den = df_istaff_clean['instructional_staff_total'].replace({0: np.nan})

df_istaff_clean['instructional_staff_long_contract_share'] = (
    df_istaff_clean['instructional_staff_long_contract'] / den
)

df_istaff_clean['instructional_staff_short_contract_share'] = (
    df_istaff_clean['instructional_staff_short_contract'] / den
)

print("df_istaff_clean dupes:", df_istaff_clean.duplicated(['unitid', 'year']).sum())

df_istaff_clean dupes: 0


In [14]:
df_cip_clean = df_cip_completers.copy()

df_cip_clean = df_cip_clean.rename(columns={
    'unitid': 'unit_id',
    'C2020_A.First or Second Major': 'major_order',
    'C2020_A.CIP Code -  2020 Classification': 'cip_code',
    'C2020_A.Award Level code': 'award_level_code',
    'C2020_A.Grand total': 'program_completers'
})

# Keep only needed columns
df_cip_clean = df_cip_clean[
    ['unit_id', 'year', 'major_order', 'cip_code', 'award_level_code', 'program_completers']
].copy()

df_cip_clean['major_order'] = df_cip_clean['major_order'].astype(str).str.strip().str.lower()

df_cip_clean = df_cip_clean[
    df_cip_clean['major_order'].eq('first major')
].copy()

In [15]:
print(sorted(df_cip_clean['award_level_code'].dropna().unique()))

["Associate's degree", "Bachelor's degree", 'Certificates of at least 1 but less than 2 years', 'Certificates of at least 12 weeks but less than 1 year', 'Certificates of at least 2 but less than 4 years', 'Certificates of less than 1 year', 'Certificates of less than 12 weeks', "Doctor's degree - other", "Doctor's degree - professional practice", "Doctor's degree - research/scholarship", "Master's degree", "Post-master's certificate", 'Postbaccalaureate certificate']


In [16]:
award_to_credential = {
    # undergraduate certificates
    'Certificates of less than 12 weeks': 1,
    'Certificates of less than 1 year': 1,
    'Certificates of at least 12 weeks but less than 1 year': 1,
    'Certificates of at least 1 but less than 2 years': 1,
    'Certificates of at least 2 but less than 4 years': 1,

    # degrees
    "Associate's degree": 2,
    "Bachelor's degree": 3,

    # post-bacc
    'Postbaccalaureate certificate': 4,

    # graduate
    "Master's degree": 5,

    # doctoral
    "Doctor's degree - research/scholarship": 6,
    "Doctor's degree - other": 6,

    # first professional
    "Doctor's degree - professional practice": 7,

    # graduate/professional certificate
    "Post-master's certificate": 8,
}

df_cip_clean['credential_level'] = (
    df_cip_clean['award_level_code']
    .astype(str)
    .str.strip()
    .map(award_to_credential)
)

unmapped = sorted(
    df_cip_clean.loc[df_cip_clean['credential_level'].isna(), 'award_level_code']
    .dropna()
    .astype(str)
    .unique()
)
print(unmapped)

df_cip_clean['program_completers'] = pd.to_numeric(
    df_cip_clean['program_completers'], errors='coerce'
)

df_cip_clean=df_cip_clean.drop(columns="award_level_code")

[]


In [17]:
def to_cip4(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.replace('.', '')
    # keep only digits
    s = ''.join(ch for ch in s if ch.isdigit())
    if len(s) >= 4:
        return s[:4]
    return np.nan

df_cip_clean['code'] = df_cip_clean['cip_code'].apply(to_cip4)

df_cip_clean = (
    df_cip_clean
    .groupby(["unit_id", "year", "code", "credential_level"], as_index=False)
    .agg({
        "program_completers": "sum"
    })
)

school_total = (
    df_cip_clean
    .groupby(["unit_id", "year"])["program_completers"]
    .transform("sum")
    .replace({0: np.nan})
)

df_cip_clean["program_completers_log"] = np.log1p(df_cip_clean["program_completers"])
df_cip_clean["program_completer_share_within_school"] = (
    df_cip_clean["program_completers"] / school_total
)


In [18]:
df_cip_clean.head(30)

,unit_id,year,code,credential_level,program_completers,program_completers_log,program_completer_share_within_school
0,132374,2020,1101,1,14,2.708050,0.016588
1,132374,2020,1102,1,10,2.397895,0.011848
2,132374,2020,1108,1,40,3.713572,0.047393
3,132374,2020,1109,1,34,3.555348,0.040284
4,132374,2020,1205,1,47,3.871201,0.055687
5,132374,2020,1503,1,0,0.000000,0.000000
6,132374,2020,1513,1,21,3.091042,0.024882
7,132374,2020,2203,1,20,3.044522,0.023697
8,132374,2020,4602,1,12,2.564949,0.014218
9,132374,2020,4603,1,33,3.526361,0.039100


In [19]:
from functools import reduce

# 1) school-level tables
school_tables = [
    df_sal_clean,
    df_gen_clean,
    df_istaff_clean,
    # df_gr_clean, etc.
]

school_driver_df = reduce(
    lambda left, right: left.merge(right, on=["unitid", "year"], how="outer"),
    school_tables
)

# standardize unit id name
school_driver_df = school_driver_df.rename(columns={"unitid": "unit_id"})

# 2) program-level table
program_driver_df = df_cip_clean.copy()

# 3) final driver table at program grain
driver_df = program_driver_df.merge(
    school_driver_df,
    on=["unit_id", "year"],
    how="left"
)

In [20]:
school_dupes = school_driver_df.duplicated(["unit_id", "year"]).sum()
program_dupes = program_driver_df.duplicated(["unit_id", "year", "code", "credential_level"]).sum()

print("school dupes on unit_id/year:", school_dupes)
print("program dupes on full program key:", program_dupes)

school dupes on unit_id/year: 0
program dupes on full program key: 0


In [21]:
save(driver_df,file_type="ipeds",clean=1, file_name="ipeds_drivers")

In [22]:
driver_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 6909 entries, 0 to 6908
Data columns (total 31 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   unit_id                                   6909 non-null   int64  
 1   year                                      6909 non-null   int64  
 2   code                                      6909 non-null   object 
 3   credential_level                          6909 non-null   int64  
 4   program_completers                        6909 non-null   int64  
 5   program_completers_log                    6909 non-null   float64
 6   program_completer_share_within_school     6906 non-null   float64
 7   instruction_salary_pct                    0 non-null      float64
 8   academic_support_salary_pct               0 non-null      float64
 9   student_services_salary_pct               0 non-null      float64
 10  research_salary_pct                 